In [1]:
# ------------------- IMPORTS -------------------
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import logging

C:\Anaconda3\envs\genai_translation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------- LOGGER SETUP -------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [3]:
# ------------------- PATH SETUP -------------------
BASE_DIR = r"E:\HOPE\AI Course Tamil\translation_project"
DATA_DIR = os.path.join(BASE_DIR, "data")
INPUT_FILE = os.path.join(DATA_DIR, "preprocessed_dataset.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "context_analysis_dataset.csv")

# ------------------- FILE CHECK -------------------
if not os.path.exists(INPUT_FILE):
    logger.error(f"Preprocessed file not found at: {INPUT_FILE}. Please run 1_trans_preprocessing.ipynb first.")
else:
    # ------------------- LOAD DATA -------------------
    df = pd.read_csv(INPUT_FILE, encoding="utf-8")
    logger.info(f" Loaded preprocessed dataset: {INPUT_FILE}")

2025-11-03 15:25:05,744 | INFO |  Loaded preprocessed dataset: E:\HOPE\AI Course Tamil\translation_project\data\preprocessed_dataset.csv


In [4]:
# ------------------- MODEL LOADING -------------------
model_name = "bhadresh-savani/distilbert-base-uncased-emotion"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

C:\Anaconda3\envs\genai_translation\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--bhadresh-savani--distilbert-base-uncased-emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

In [8]:
# ------------------- ANALYZE CONTEXT -------------------

def detect_emotion(text):
    # Handle empty or invalid text safely
    if not isinstance(text, str) or text.strip() == "":
        return "unknown"
    
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            label_id = torch.argmax(probs, dim=1).item()
            label = model.config.id2label[label_id]
        return label
    except Exception as e:
        logger.warning(f"Skipping text due to error: {e}")
        return "unknown"



In [9]:
 # Apply to dataset
df["emotion_label"] = df["input_text"].apply(detect_emotion)

In [11]:
df.head(10)

,id,input_text,target_language,human_translated_text,clean_text,detected_lang,moderation_status,emotion_label
0,1,அழுமூஞ்சியாக இருக்காதே.,en,Don’t be a cry-baby. / Don’t be a fusspot.,அழுமூஞ்சியாக இருக்காதே.,ta,approved,anger
1,2,சிணுங்குவதை நிறுத்து.,en,Stop whining. / Stop whimpering.,சிணுங்குவதை நிறுத்து.,ta,approved,anger
2,3,நீ மிகவும் குறும்புக்காரி.,en,You are very naughty.,நீ மிகவும் குறும்புக்காரி.,ta,approved,anger
3,4,இந்த பையன் மிகவும் தேட்தடச் சேயவன்.,en,This boy is very mischievous; troublesome.,இந்த பையன் மிகவும் தேட்தடச் சேயவன்.,ta,approved,joy
4,5,ஒழுங்கா இரு.,en,Behave yourself. / Keep quiet.,ஒழுங்கா இரு.,ta,approved,anger
5,6,அவள் ஒரு சகட்டிக்காரி.,en,"She is smart, bright and intelligent.",அவள் ஒரு சகட்டிக்காரி.,ta,approved,joy
6,7,அவள் சரம் வாயாடி.,en,She yaps a lot. / She is a chatterbox. / Talka...,அவள் சரம் வாயாடி.,ta,approved,joy
7,8,அவள் பிடிவாதமானவள்.,en,Stubborn; adamant.,அவள் பிடிவாதமானவள்.,ta,approved,joy
8,9,அவள் காபி தரையில் சிந்திவிட்டாள்.,en,She spilled coffee on the floor. / She messed ...,அவள் காபி தரையில் சிந்திவிட்டாள்.,ta,approved,joy
9,10,எல்லாவற்றையும் நீ தான் செய்யணும். நீ வளர்ந்துவ...,en,Help yourself. (Do it yourself.) You’ve grown up.,எல்லாவற்றையும் நீ தான் செய்யணும். நீ வளர்ந்துவ...,ta,approved,joy


In [13]:
 # ------------------- SAVE OUTPUT -------------------
os.makedirs(DATA_DIR, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
logger.info(f"✅ Context analysis completed and saved at: {OUTPUT_FILE}")

2025-11-03 15:32:08,737 | INFO | ✅ Context analysis completed and saved at: E:\HOPE\AI Course Tamil\translation_project\data\context_analysis_dataset.csv
